In [ ]:
!pip install langchain langchain-huggingface faiss-cpu datasets evaluate langchain-community -U
!pip install torch accelerate transformers

In [ ]:
!pip install langchain-core -U
!pip install langchain-classic
!pip install sentence-transformers
!pip install rouge_score
!pip install vllm
!pip install rank_bm25
!pip install -U ipywidgets

In [3]:
from datasets import load_dataset
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains  import RetrievalQA 
from transformers import pipeline
import evaluate
from langchain_core.documents import Document
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from typing import List, Dict

In [4]:
def prepare_documents(knowledge_base: Dict[int, str]) -> List[Document]:
    """Преобразует словарь с базой знаний в список объектов Document для LangChain."""
    documents = []
    for item_id, text in knowledge_base.items():
        doc = Document(page_content=text, metadata={'id': item_id})
        documents.append(doc)
    return documents

In [5]:
def get_embedding_model(model_name: str = 'intfloat/multilingual-e5-large-instruct') -> HuggingFaceEmbeddings:
    """Создает и возвращает модель для векторизации текста через обертку LangChain."""
    print(f"Инициализация модели эмбеддингов '{model_name}'...")
    model_kwargs = {'device': 'cpu'}
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs
    )
    print("Модель успешно инициализирована.")
    return embeddings

In [6]:
KNOWLEDGE_BASE = {
    101: "Оформить дебетовую карту 'Мир'",
    102: "Узнать баланс по кредитной карте",
    103: "Заблокировать банковскую карту",
    104: "Получить выписку по счету дебетовой карты",
    105: "Открыть накопительный вклад",
    106: "Консультация по ипотечному кредитованию",
}

QUERIES = [
    "как сделать карточку для денег?", # Здесь лучше справится семантика
    "заморозить карту", # Оба должны справиться
    "выписка по дебетовой карте", # Здесь важны ключевые слова "выписка" и "дебетовая"
    "как перевести деньги другу?", # Ничего не должно найтись
]

In [7]:
docs = prepare_documents(KNOWLEDGE_BASE)

# Шаг 2: Создание векторного ретривера (семантический поиск)
print("\n--- Создание семантического ретривера (FAISS) ---")
embedding_model = get_embedding_model()
vector_store = FAISS.from_documents(docs, embedding_model)

# .as_retriever() превращает векторную базу в объект Retriever
# search_kwargs={'k': 3} означает, что он будет возвращать до 3 лучших кандидатов
faiss_retriever = vector_store.as_retriever(search_kwargs={'k': 3})


--- Создание семантического ретривера (FAISS) ---
Инициализация модели эмбеддингов 'intfloat/multilingual-e5-large-instruct'...


/var/folders/0k/z_l4ql_113943rc3bncsv20r0000gn/T/ipykernel_61427/2717137243.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Модель успешно инициализирована.


In [8]:
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 3 # Также просим его вернуть до 3 кандидатов
print("BM25 retriever создан.")

BM25 retriever создан.


In [ ]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5] # Задаем равные веса для начала
)
print("EnsembleRetriever успешно создан.")

EnsembleRetriever успешно создан.


In [ ]:
USE_LLM = False

In [ ]:
if USE_LLM:

    from langchain_community.llms import VLLM
    from langchain_core.prompts import PromptTemplate

    # создаём свой системный промпт
    system_prompt = """
    You must answer the question as clearly as possible based on the documents provided by user. In your answer, add a little thought and highlight your final answer as follows:
    Final answer: "your answer". 
    """

    # шаблон, который используется в RAG
    prompt_template = """
    Documents: {context}

    Question: {question}

    """

    # собираем кастомный шаблон, добавив системный инструктаж
    full_prompt = PromptTemplate(
        input_variables=["context", "question"],
        template=system_prompt + "\n\n" + prompt_template,
    )

    llm = VLLM(
        model="Qwen/Qwen3-VL-4B-Instruct",
        temperature=0,
        trust_remote_code=True,
        tensor_parallel_size=1,
        vllm_kwargs={
            "max_model_len": 4096,
            "max_num_batched_tokens": 4096,
            "gpu_memory_utilization": 0.7,
        },
    )

        # 4️⃣ Сборка цепочки Retrieval-Augmented Generation
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type_kwargs={"prompt": full_prompt},
        retriever=ensemble_retriever,
        chain_type="stuff",
        return_source_documents=False
    )

In [ ]:
for user_query in QUERIES:
    print(f"Входящий запрос: '{user_query}'")
    
    # Получаем отсортированный список релевантных документов
    # EnsembleRetriever не возвращает score, он возвращает итоговый ранжированный список
    if USE_LLM:
        results = qa_chain.invoke(user_query)
    else:
        results = ensemble_retriever.invoke(user_query)
    
    if results:
        # Лучший результат — первый в списке
        best_match = results[0]
        matched_item = best_match.page_content
        item_id = best_match.metadata['id']
        
        print(f"  -> Стандартизовано в: '{matched_item}' (ID: {item_id})\n")
        
        # Для наглядности посмотрим на весь топ результатов от гибридного поиска
        print("  Весь топ от гибридного поиска:")
        for i, doc in enumerate(results):
            print(f"    {i+1}. '{doc.page_content}' (ID: {doc.metadata['id']})")
        print("-" * 20)

    else:
        print("  -> Подходящий эталон не найден.\n")

Входящий запрос: 'как сделать карточку для денег?'
  -> Стандартизовано в: 'Получить выписку по счету дебетовой карты' (ID: 104)

  Весь топ от гибридного поиска:
    1. 'Получить выписку по счету дебетовой карты' (ID: 104)
    2. 'Консультация по ипотечному кредитованию' (ID: 106)
    3. 'Оформить дебетовую карту 'Мир'' (ID: 101)
    4. 'Открыть накопительный вклад' (ID: 105)
    5. 'Заблокировать банковскую карту' (ID: 103)
--------------------
Входящий запрос: 'заморозить карту'
  -> Стандартизовано в: 'Заблокировать банковскую карту' (ID: 103)

  Весь топ от гибридного поиска:
    1. 'Заблокировать банковскую карту' (ID: 103)
    2. 'Оформить дебетовую карту 'Мир'' (ID: 101)
    3. 'Консультация по ипотечному кредитованию' (ID: 106)
    4. 'Узнать баланс по кредитной карте' (ID: 102)
--------------------
Входящий запрос: 'выписка по дебетовой карте'
  -> Стандартизовано в: 'Узнать баланс по кредитной карте' (ID: 102)

  Весь топ от гибридного поиска:
    1. 'Узнать баланс по кредит